# Analysis dataset EDA: `analysis_measurement`

Explores the modeling table written by `build_analysis_dataset.py` (one row per pre/post measurement of each eligible dam-station pair; columns and design choices in `docs/warehouse_spec.md`, section 3.11), before modeling:

1. Overview: size, context coverage, per-parameter summary
2. Balance: pre vs post samples, years from removal, sampling months
3. Value distributions
4. Seasonality
5. Before vs after: raw and with the seasonal cycle removed
6. Event study: change by year from removal, downstream vs upstream
7. Dam and watershed covariates
8. Climate vs water quality
9. A first simple model per parameter (preview, not final)

Reads the Parquet file directly (no `warehouse.duckdb` connection), so it never locks out Slurm jobs.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from etl.common import wq_rules
from etl.common.config import load_config
from etl.common.warehouse import warehouse_dir

CASE_STUDY = "pilot_random50_seed7"
cfg = load_config(CASE_STUDY, project_root=ROOT)
WAREHOUSE = warehouse_dir(cfg)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

df = pd.read_parquet(WAREHOUSE / "analysis_measurement.parquet")

# One label per parameter x fraction ("phosphorus (Total)", "water_temp").
df["series"] = df["parameter_std"].where(
    df["sample_fraction_std"].isna(),
    df["parameter_std"] + " (" + df["sample_fraction_std"].fillna("") + ")",
)

# Skewed, always-positive parameters are looked at on a log10 scale (y).
# Zeros are floored at 0.001 so the log is defined.
LOG_PARAMETERS = {"turbidity", "specific_conductance", "nitrate", "ammonia",
                  "orthophosphate", "phosphorus", "tds"}
df["log_scale"] = df["parameter_std"].isin(LOG_PARAMETERS)
df["y"] = np.where(df["log_scale"], np.log10(df["result_value_std"].clip(lower=1e-3)),
                   df["result_value_std"])
df["post"] = (df["period"] == "post").astype(int)
df["pair"] = df["AR_ID"] + " | " + df["station_id"]

SERIES = df["series"].value_counts().index.tolist()  # most data first


def y_label(series):
    parameter = series.split(" (")[0]
    unit = wq_rules.STANDARD_UNITS.get(parameter, "")
    return f"log10 {unit}" if parameter in LOG_PARAMETERS else unit


def facet_axes(n, ncols=4, width=4, height=3):
    """A grid of n axes, extra ones hidden."""
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(width * ncols, height * nrows), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat


print("Case study:", cfg.name)
print(f"Rows: {len(df):,}; dams: {df['AR_ID'].nunique()}; stations: {df['station_id'].nunique()}; "
      f"dam-station pairs: {df['pair'].nunique()}; series: {len(SERIES)}")

## 1. Overview

In [ ]:
# Share of rows with each column filled in (context columns are left null, not dropped).
coverage = df.notna().mean().mul(100).round(1).sort_values()
coverage[coverage < 100].to_frame("% non-null")

In [ ]:
# Per parameter: size, balance and censoring.
summary = df.groupby("series").agg(
    rows=("measurement_id", "size"),
    dams=("AR_ID", "nunique"),
    stations=("station_id", "nunique"),
    pre=("post", lambda p: int((p == 0).sum())),
    post=("post", "sum"),
    pct_censored=("is_censored", lambda c: round(100 * c.mean(), 1)),
    pct_downstream=("side", lambda s: round(100 * (s == "downstream").mean(), 1)),
    pct_nwis=("provider", lambda p: round(100 * (p == "NWIS").mean(), 1)),
)
summary["unit"] = [y_label(s).replace("log10 ", "") for s in summary.index]
summary.sort_values("rows", ascending=False)

In [ ]:
# Dam-station pairs per parameter by side: downstream pairs are the main
# comparison; upstream pairs at the same dam are possible controls.
(df.drop_duplicates(["pair", "series"])
   .pivot_table(index="series", columns="side", values="pair", aggfunc="count", fill_value=0)
   .loc[SERIES])

## 2. Balance

If pre and post samples differ in number, season or provider, a raw before/after comparison is biased. These cells show how balanced the data is.

In [ ]:
# Samples per pair, pre vs post (log scale). Points far from the diagonal are unbalanced pairs.
pairs = (df.groupby(["pair", "series", "side"])
           .agg(n_pre=("post", lambda p: (p == 0).sum()), n_post=("post", "sum"))
           .reset_index())
fig, axes = facet_axes(len(SERIES))
for ax, series in zip(axes, SERIES):
    rows = pairs[pairs["series"] == series]
    sns.scatterplot(data=rows, x="n_pre", y="n_post", hue="side", ax=ax, s=20, legend=ax is axes[0])
    lim = [1, max(rows[["n_pre", "n_post"]].max().max(), 10) * 1.5]
    ax.plot(lim, lim, color="gray", lw=1, ls="--")
    ax.set(xscale="log", yscale="log", title=series)
fig.tight_layout()

In [ ]:
# Rows by year from removal (removal year 0 is excluded by design).
counts = df.pivot_table(index="series", columns="years_from_removal", values="y",
                        aggfunc="count", fill_value=0).loc[SERIES]
plt.figure(figsize=(12, 0.4 * len(SERIES) + 1))
sns.heatmap(counts, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Rows by year from removal")

In [ ]:
# Sampling months, pre vs post: if post samples are more summer-heavy than
# pre, seasonality alone shifts the raw comparison.
months = (df.groupby(["series", "period"])["month"].value_counts(normalize=True)
            .rename("share").reset_index())
fig, axes = facet_axes(len(SERIES))
for ax, series in zip(axes, SERIES):
    sns.barplot(data=months[months["series"] == series], x="month", y="share", hue="period",
                hue_order=["pre", "post"], ax=ax)
    ax.set_title(series)
    if ax is not axes[0]:
        ax.get_legend().remove()
fig.tight_layout()

In [ ]:
# Provider mix pre vs post (NWIS = USGS, STORET = states and others): a switch
# in who samples can look like a change in water quality.
(df.groupby(["series", "period"])["provider"].value_counts(normalize=True)
   .mul(100).round(1).unstack(fill_value=0))

## 3. Value distributions

In [ ]:
fig, axes = facet_axes(len(SERIES))
for ax, series in zip(axes, SERIES):
    rows = df[df["series"] == series]
    sns.histplot(data=rows, x="y", hue="period", hue_order=["pre", "post"], bins=40,
                 element="step", stat="density", common_norm=False, ax=ax, legend=ax is axes[0])
    ax.set(title=series, xlabel=y_label(series))
fig.tight_layout()

## 4. Seasonality

Why monthly climate matters: how much each parameter moves through the year, compared with any before/after shift.

In [ ]:
fig, axes = facet_axes(len(SERIES))
for ax, series in zip(axes, SERIES):
    sns.boxplot(data=df[df["series"] == series], x="month", y="y", ax=ax,
                color="lightsteelblue", fliersize=1)
    ax.set(title=series, ylabel=y_label(series))
fig.tight_layout()

In [ ]:
# Water temperature vs that month's upstream air temperature.
rows = df[df["parameter_std"] == "water_temp"].dropna(subset=["air_temp_mean_c"])
ax = sns.scatterplot(data=rows, x="air_temp_mean_c", y="y", hue="period", hue_order=["pre", "post"],
                     s=8, alpha=0.4)
ax.plot([-10, 30], [-10, 30], color="gray", ls="--", lw=1)
ax.set(xlabel="Monthly mean air temperature, upstream (°C)", ylabel="Water temperature (°C)")
print(f"Correlation: {rows['air_temp_mean_c'].corr(rows['y']):.2f}")

## 5. Before vs after

**Anomaly** = value minus the station's average for that parameter and calendar month (over both periods). It removes each station's seasonal cycle and level, so a before/after shift stands out. It's a quick EDA device; the models in section 9 use month terms instead.

In [ ]:
df["anomaly"] = df["y"] - df.groupby(["station_id", "series", "month"])["y"].transform("mean")

# One point per downstream pair: median before vs median after (raw values).
medians = (df[df["side"] == "downstream"]
           .groupby(["pair", "series", "period"])["y"].median().unstack().dropna())
fig, axes = facet_axes(len(SERIES))
for ax, series in zip(axes, SERIES):
    if series not in medians.index.get_level_values("series"):
        continue
    rows = medians.xs(series, level="series")
    ax.scatter(rows["pre"], rows["post"], s=15)
    lim = [rows.min().min(), rows.max().max()]
    ax.plot(lim, lim, color="gray", ls="--", lw=1)
    ax.set(title=f"{series} (n={len(rows)})", xlabel="median before", ylabel="median after")
fig.suptitle("Downstream pairs: median before vs after (raw; points above the line went up)")
fig.tight_layout()

In [ ]:
# Change per pair after removing the seasonal cycle: mean anomaly after - before.
change = (df.groupby(["pair", "AR_ID", "series", "side", "period"])["anomaly"].mean()
            .unstack().dropna())
change["change"] = change["post"] - change["pre"]
change = change.reset_index()

plt.figure(figsize=(12, 0.45 * len(SERIES) + 1))
sns.boxplot(data=change, y="series", x="change", hue="side", order=SERIES,
            hue_order=["downstream", "upstream"], fliersize=2)
plt.axvline(0, color="k", lw=1)
plt.xlabel("Mean anomaly after − before (units of y; log10 for skewed parameters)")
plt.title("Per-pair change, seasonal cycle removed")

# Share of downstream pairs that went up, per parameter.
(change[change["side"] == "downstream"].groupby("series")["change"]
   .agg(pairs="size", median_change="median", pct_increased=lambda c: round(100 * (c > 0).mean())))

## 6. Event study

Mean anomaly by year from removal. A removal effect would show as a step (or trend) after year 0 at downstream stations but not upstream ones.

In [ ]:
fig, axes = facet_axes(len(SERIES))
for ax, series in zip(axes, SERIES):
    rows = df[df["series"] == series]
    sns.lineplot(data=rows, x="years_from_removal", y="anomaly", hue="side",
                 hue_order=["downstream", "upstream"], errorbar=("ci", 95), marker="o",
                 ax=ax, legend=ax is axes[0])
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.axhline(0, color="gray", lw=0.5)
    ax.set(title=series, ylabel=f"anomaly ({y_label(series)})")
fig.tight_layout()

## 7. Dam and watershed covariates

One row per dam-station pair (these columns don't vary within a pair).

In [ ]:
COVARIATES = ["dam_height_ft", "dam_age_at_removal", "upstream_area_km2", "network_distance_km",
              "pct_forest", "pct_developed", "pct_agriculture", "pct_wetlands", "pct_water"]
pair_table = df.drop_duplicates("pair")[["pair", "AR_ID", "side", "removal_year",
                                         "land_cover_year", "watershed_truncated"] + COVARIATES]
print(f"Pairs: {len(pair_table)}; land cover from after removal: "
      f"{int((pair_table['land_cover_year'] >= pair_table['removal_year']).sum())}; "
      f"truncated watersheds: {int(pair_table['watershed_truncated'].sum())}")
pair_table[COVARIATES].describe().round(1)

In [ ]:
fig, axes = facet_axes(len(COVARIATES), ncols=5, width=3.2, height=2.6)
for ax, column in zip(axes, COVARIATES):
    values = pair_table[column].dropna()
    if column in ("upstream_area_km2", "network_distance_km"):
        values = np.log10(values.clip(lower=0.01))
        column = f"log10 {column}"
    sns.histplot(values, bins=20, ax=ax)
    ax.set(title=column, xlabel="")
fig.tight_layout()

In [ ]:
# Correlations between covariates (Spearman): strongly correlated pairs
# shouldn't both go into a small model.
corr = pair_table[COVARIATES].corr(method="spearman")
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, mask=np.triu(np.ones_like(corr, bool), 1))
plt.title("Covariate correlations (Spearman), one row per pair")

In [ ]:
# Does the per-pair change (section 5) relate to the covariates? Downstream pairs only.
# FOCUS: one series at a time; change it to look at others.
FOCUS = SERIES[0]
rows = change[(change["series"] == FOCUS) & (change["side"] == "downstream")].merge(
    pair_table[["pair"] + COVARIATES], on="pair")
fig, axes = facet_axes(len(COVARIATES), ncols=5, width=3.2, height=2.6)
for ax, column in zip(axes, COVARIATES):
    sns.scatterplot(data=rows, x=column, y="change", ax=ax, s=20)
    ax.axhline(0, color="gray", lw=0.5)
    if column in ("upstream_area_km2", "network_distance_km"):
        ax.set_xscale("log")
fig.suptitle(f"{FOCUS}: per-pair change vs covariates (n={len(rows)})")
fig.tight_layout()

## 8. Climate vs water quality

In [ ]:
# Spearman correlation of each parameter with the climate columns.
CLIMATE = ["air_temp_mean_c", "precip_mm", "precip_prev_month_mm"]
climate_corr = pd.DataFrame({
    series: df.loc[df["series"] == series, ["y"] + CLIMATE].corr(method="spearman")["y"][CLIMATE]
    for series in SERIES
}).T
plt.figure(figsize=(6, 0.4 * len(SERIES) + 1))
sns.heatmap(climate_corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Correlation with upstream climate (Spearman)")

In [ ]:
# Turbidity vs rain the month before (where turbidity is present).
rows = df[df["parameter_std"] == "turbidity"].dropna(subset=["precip_prev_month_mm"])
if len(rows):
    ax = sns.scatterplot(data=rows, x="precip_prev_month_mm", y="y", s=8, alpha=0.4)
    ax.set(xlabel="Precipitation the month before (mm)", ylabel="log10 turbidity")

## 9. A first simple model per parameter (preview)

Two linear mixed models per parameter, with a random intercept per dam-station pair (repeated samples at one station aren't independent). `y` is log10 for skewed parameters, so `10 ** effect − 1` is the proportional change.

- **Before/after (downstream only):** `y ~ post + C(month) + air_temp_mean_c + precip_prev_month_mm`. `post` is the average change after removal, given season and weather.
- **BACI (dams with upstream controls):** adds `downstream` and `post:downstream`. The interaction is the change downstream beyond any change upstream (e.g. a regional trend).

Exploratory only: no censoring model, no distance or dam covariates, and small samples. Rows with missing climate are dropped.

In [ ]:
def fit(formula, rows, term):
    """Coefficient, 95% CI and p-value of one term, or None if the fit fails."""
    try:
        result = smf.mixedlm(formula, rows, groups=rows["pair"]).fit(reml=True, method="lbfgs")
    except Exception as exc:  # singular fits etc. with very little data
        return {"error": str(exc)[:60]}
    low, high = result.conf_int().loc[term]
    return {"effect": result.params[term], "ci_low": low, "ci_high": high,
            "p": result.pvalues[term], "rows": int(result.nobs), "pairs": rows["pair"].nunique()}


COVARIATE_TERMS = "C(month) + air_temp_mean_c + precip_prev_month_mm"
results = []
for series in SERIES:
    rows = df[df["series"] == series].dropna(subset=["air_temp_mean_c", "precip_prev_month_mm", "side"])
    downstream = rows[rows["side"] == "downstream"]
    if downstream["pair"].nunique() >= 3:
        results.append({"series": series, "model": "before/after (downstream)",
                        **fit(f"y ~ post + {COVARIATE_TERMS}", downstream, "post")})

    # BACI: dams that have both a downstream and an upstream pair.
    sides = rows.groupby("AR_ID")["side"].nunique()
    baci = rows[rows["AR_ID"].isin(sides[sides == 2].index)].assign(
        downstream=lambda r: (r["side"] == "downstream").astype(int))
    if baci["AR_ID"].nunique() >= 2:
        results.append({"series": series, "model": "BACI (post x downstream)",
                        **fit(f"y ~ post * downstream + {COVARIATE_TERMS}", baci, "post:downstream")})

results = pd.DataFrame(results)
results["log_scale"] = results["series"].str.split(" (", regex=False).str[0].isin(LOG_PARAMETERS)
# For log10 parameters: the effect as a % change.
results["pct_change"] = np.where(results["log_scale"], (10 ** results["effect"] - 1) * 100, np.nan)
results.round(4)

In [ ]:
# Effects with 95% CIs (units of y; log10 for skewed parameters). Intervals
# that cross 0 are consistent with no change.
ok = results.dropna(subset=["effect"])
fig, axes = plt.subplots(1, 2, figsize=(14, 0.4 * len(SERIES) + 1.5), sharey=False)
for ax, (model, rows) in zip(axes, ok.groupby("model")):
    ax.errorbar(rows["effect"], rows["series"],
                xerr=[rows["effect"] - rows["ci_low"], rows["ci_high"] - rows["effect"]],
                fmt="o", capsize=3)
    ax.axvline(0, color="k", lw=1)
    ax.set(title=model, xlabel="effect (95% CI)")
fig.tight_layout()

## Findings

_Notes from running this notebook: imbalances to handle, parameters with signal, covariates worth keeping, and what to change before modeling or scaling up._

-